# Imports

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import collections
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from cycler import cycler
import numpy as np
import pylab
import FlowCal
import copy
from matplotlib.lines import Line2D
import warnings
import corner
from matplotlib.colors import ListedColormap

import emcee

# Set default parameters for plots
sns.set_context("talk", font_scale=1, rc={"lines.linewidth": 2.0, 'lines.markersize': 5})
sns.set_style("ticks")
sns.set_style({"xtick.direction": "in","ytick.direction": "in"})
mpl.rc('axes', prop_cycle=(cycler('color', ['r', 'k', 'b','g','y','m','c']) ))
mpl.rc('text', usetex=False)
#mpl.rc('font',**{'family':'sans-serif','sans-serif':['HelveticaNeueThin']})
#plt.tick_params(top=False, bottom=False, right=False, left=False)
tw = 1.5
sns.set_style({"xtick.major.size": 6, "ytick.major.size": 6,
               "xtick.minor.size": 4, "ytick.minor.size": 4,
               'axes.labelsize': 24,
               'xtick.major.width': tw, 'xtick.minor.width': tw,
               'ytick.major.width': tw, 'ytick.minor.width': tw})

mpl.rc('xtick', labelsize=18) 
mpl.rc('ytick', labelsize=18)
mpl.rc('axes', linewidth=1.75)
plt.gcf().subplots_adjust(bottom=0.15)
sns.set_style({'axes.labelsize': 24, 'axes.titlesize': 24})

# %matplotlib inline
# warnings.filterwarnings(action='once')
warnings.filterwarnings('ignore')

<Figure size 640x480 with 0 Axes>

# Functions

In [2]:
# gRNA properties

# This is a new PAM dict that shows the position of the central
# bp of the 33bp dCas9 footprint, which is the 5th bp upstream of PAM.
# Positions are given in absolute values, as in 0 is immediately before +1.
# To convert to genomic position, subtract 1 from all negative values.
PAM_dict = {'g1': -27,
            'g2': -22,
            'g3': -5,
            'g4': 12,
            'g5': 23,
            'g6': 25,
            'g8': 43,
            'g9': 50,
            'g10': 67,
            'g11': 101,
            'g12': 156,
            'g13': 160,
           }

strand_dict = {'g1': 'right',
               'g2': 'left',
               'g3': 'right',
               'g4': 'right',
               'g5': 'left',
               'g6': 'right',
               'g8': 'right',
               'g9': 'left',
               'g10': 'left',
               'g11': 'right',
               'g12': 'right',
               'g13': 'left',
              }

def strand_to_footprint(PAM, strand):
    '''
    PAM is an integer.
    strand is a string that is 'right' or 'left'.
    '''
    if strand == 'right':
        rightwindow = PAM + 8 # 9bp overhang beyond gRNA. N in NGG already included.
        leftwindow = PAM - 20 - 1 #1bp overhang beyond gRNA.
    elif strand == 'left':
        rightwindow = PAM + 20 + 1
        leftwindow = PAM - 8
    else:
        raise('Problem with strand')
    return leftwindow, rightwindow

In [3]:
# Helper functions

def Hill_rep_fun(x, params):
    '''
    Hill repression function that returns y for input x.
    Written in numerically stable form.
    '''
    b, a, K, n = params
    return b / (1 + np.exp(n*(np.log(x) - np.log(K)))) + a

def Hill_act_fun(x, params):
    '''
    Hill activation function that returns y for input x.
    Written in numerically stable form.
    '''
    b, a, K, n = params
    return b * np.exp(n*(np.log(x) - np.log(K))) / (1 + np.exp(n*(np.log(x) - np.log(K)))) + a

def symlog_transform(x, linthresh=1.0):
    return np.sign(x) * (np.log1p(np.abs(x / linthresh)))
def symlog_inverse(y, linthresh=1.0):
    return np.sign(y) * linthresh * (np.expm1(np.abs(y)))

def log_lognormal(x, mu, sigma):
    '''
    Returns the log prior for a lognormal distribution.
    Takes x as input in non-log-scaled value.
    To obtain a median of M for the distribution, must input mu = ln(M).
    '''
    return -np.log(x) - np.log(sigma) - 0.5 * np.log(2 * np.pi) - ((np.log(x) - mu) ** 2) / (2 * sigma**2)


In [4]:
# functions for emcee

def log_prior(theta, prior_params):     
    '''
    Lognormal priors for all four parameters.
    '''
    b, a, K, n = theta
    b_params, a_params, K_params, n_params = prior_params
    mub, sigmab = b_params
    mua, sigmaa = a_params
    muK, sigmaK = K_params
    mun, sigman = n_params
    
    # log-normal prior on beta
    logbprior = -np.log(b) - np.log(sigmab) - 0.5 * np.log(2 * np.pi) - ((np.log(b) - mub) ** 2) / (2 * sigmab**2)
    # log-normal prior on alpha
    logaprior = -np.log(a) - np.log(sigmaa) - 0.5 * np.log(2 * np.pi) - ((np.log(a) - mua) ** 2) / (2 * sigmaa**2)
    # log-normal prior on K
    logKprior = -np.log(K) - np.log(sigmaK) - 0.5 * np.log(2 * np.pi) - ((np.log(K) - muK) ** 2) / (2 * sigmaK**2)
    # log-normal prior on n
    lognprior =  -np.log(n) - np.log(sigman) - 0.5 * np.log(2 * np.pi) - ((np.log(n) - mun) ** 2) / (2 * sigman**2)
    
    return logbprior + logaprior + logKprior + lognprior

def log_likelihood(theta, x, y):
    '''
    Model is a Hill repression function, with output in ln() scale.
    This requires data, y, to also be in ln() scale.
    '''
    b, a, K, n = theta

    model = np.log(b / (1 + np.exp(n*(np.log(x) - np.log(K)))) + a) # <- this is ln(Hill rep function), written in numerically stable form
    
    cost = - np.linalg.norm(y - model) # <- Make sure the model and the data are appropriately log-transformed   
    cost_progress.append(cost)
    return cost


def log_probability(theta, x, y, prior_params):
    lp = log_prior(theta, prior_params)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, x, y)

# Unpickle gated FCS files

In [5]:
with open('./pickles/241130_fcs_gated_dict.pkl', 'rb') as f:
    fcs_gated_dict_1 = pickle.load(f)
f.close()

with open('./pickles/241205_fcs_gated_dict.pkl', 'rb') as f:
    fcs_gated_dict_2 = pickle.load(f)
f.close()

with open('./pickles/241208_fcs_gated_dict.pkl', 'rb') as f:
    fcs_gated_dict_3 = pickle.load(f)
f.close()

# Parameter Estimation


## Run MCMC

In [6]:
fcs_dict_ls = [fcs_gated_dict_1,
          fcs_gated_dict_2,
          fcs_gated_dict_3,
         ]

medias = ['M9', 'CL']
reps = [1,2,3]
grnas = ['g1','g2','g3','g4','g5','g6',
         'g8','g9','g10','g11','g12','g13','g14']

# atc induction conditions for inducing dCas9 in each experimental condition
atcs_cl_1 = ['Blank','0','0.02','0.06','0.2','0.43','0.93','2','6','20','60','200']
atcs_m9_1 = ['Blank','0','0.02','0.06','0.2','0.43','0.93','2','6','20','60','200']
atcs_cl_2 = ['Blank','0','0.02','0.06','0.2','0.43','0.93','2','6','20','60','200']
atcs_m9_2 = ['Blank','0','0.02','0.2','0.43','0.93','2','3.5','6.3','11.3','20','200']
atcs_cl_3 = ['Blank','0','0.02','0.06','0.2','0.43','0.93','2','6','20','60','200']
atcs_m9_3 = ['Blank','0','0.2','0.43','0.93','2','2.6','3.5','6.3','11.3','20','200']

atcs_ls = [[atcs_m9_1, atcs_cl_1],
            [atcs_m9_2, atcs_cl_2],
            [atcs_m9_3, atcs_cl_3],
          ]

plot_inds_cl_1 = [0, 0.02, 0.06, 0.2, 0.43, 0.93, 2, 6, 20, 60, 200]
plot_inds_m9_1 = [0, 0.02, 0.06, 0.2, 0.43, 0.93, 2, 6, 20, 60, 200]
plot_inds_cl_2 = [0, 0.02, 0.06, 0.2, 0.43, 0.93, 2, 6, 20, 60, 200]
plot_inds_m9_2 = [0, 0.02, 0.2, 0.43, 0.93, 2, 3.5, 6.3, 11.3, 20, 200]
plot_inds_cl_3 = [0, 0.02, 0.06, 0.2, 0.43, 0.93, 2, 6, 20, 60, 200]
plot_inds_m9_3 = [0, 0.2, 0.43, 0.93, 2, 2.6, 3.5, 6.3, 11.3, 20, 200]

plot_inds_ls = [[plot_inds_m9_1, plot_inds_cl_1],
                [plot_inds_m9_2, plot_inds_cl_2],
                [plot_inds_m9_3, plot_inds_cl_3],
               ]

In [7]:
nwalkers = 32
ndim = 4
chain_length = 10000

sigmaB = 0.5
sigmaA = 1
sigmaK = 1
sigmaN = 0.5
muK = np.log(0.1) + sigmaK**2
muN = np.log(1.0) + sigmaN**2

        
for grna in grnas:
    for m in range(len(medias)):
        media = medias[m]

        med_gfps = []
        med_rfps = []
        unrep_gfps = []
        rep_gfps = []
        for rep in reps:
            fcs_gated_dict = fcs_dict_ls[rep-1]
            plot_inds = plot_inds_ls[rep-1][m]
            atcs = atcs_ls[rep-1][m]
            for n in range(len(plot_inds)):
                fcs = fcs_gated_dict[media+'_'+grna+'_'+atcs[n+1]]
                gfps = np.array(fcs[:,5])
                rfps = np.array(fcs[:,7])
                med_gfps.append(np.median(gfps))
                med_rfps.append(np.median(rfps))
                if n == 0:
                    unrep_gfps.append(np.median(gfps))
                if n == len(plot_inds)-1:
                    rep_gfps.append(np.median(gfps))

        # band-aid fix: set negative values to 1e-6 for fitting
        med_gfps = np.array(med_gfps)
        med_rfps = np.array(med_rfps)
        
        med_gfps[med_gfps < 0] = 1e-6
        med_rfps[med_rfps < 0] = 1e-6

        # prepare for emcee

        # take geomean of repressed GFP values from each replicate
        alpha_guess = stats.gmean(rep_gfps)
        # take geomean of unrepressed GFP values from each replicate
        unrepressed_guess = stats.gmean(unrep_gfps)
        
        beta_guess = unrepressed_guess - alpha_guess
        # what if beta is negative? for now, bandaid fix of set it to 1e-6
        if beta_guess < 0:
            beta_guess = 1e-6

        # ln-scale everything for comparison to ln-scaled model
        med_gfps = np.log(med_gfps)
        alpha_guess = np.log(alpha_guess)
        beta_guess = np.log(beta_guess)
        
        # set prior scale parameters
        muB = beta_guess + sigmaB**2 # <- note that the 'guess' values are already in ln() scale.
        muA = alpha_guess + sigmaA**2
        theta0 = np.exp(np.array([beta_guess, alpha_guess, muK, muN])) # <- initial parameter position is in standard scale
        prior_params = np.array([np.array([muB, sigmaB]),
                                 np.array([muA, sigmaA]),
                                 np.array([muK, sigmaK]),
                                 np.array([muN, sigmaN]),
                                ])

        # random initialization of walkers
        pos = theta0 * (1 + 1e-4 * np.random.randn(nwalkers, ndim))
        pos = 10**(np.log10(theta0) + np.random.randn(nwalkers,ndim)*1e-4) 

        not_long_enough = True
        length_factor = 1        
        while not_long_enough:        
            cost_progress = []
            print('Starting MCMC for '+grna+' , '+media+' for '+str(chain_length*length_factor)+' steps...')
            sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability, args=(med_rfps, med_gfps, prior_params))
            sampler.run_mcmc(pos, chain_length*length_factor, progress=True);

            autocorr_vec = sampler.get_autocorr_time()
            max_tau = np.max(autocorr_vec)
            if max_tau*50 <= chain_length*length_factor:
                not_long_enough = False
            else:
                length_factor *= 10

            if length_factor >= 100:
                print('******* Chain length too long! aborting. ********')
                not_long_enough = False

        # pickle the sampler output
        filename = grna + '_' + media + '_' + str(nwalkers) + '_' + str(chain_length*length_factor) + '.pkl'
        with open('./MCMC_outputs/'+filename, 'wb') as f:
            pickle.dump(sampler, f)
        f.close()


Starting MCMC for g1 , M9 for 10000 steps...


100%|███████████████████████████████████| 10000/10000 [00:05<00:00, 1779.67it/s]


Starting MCMC for g1 , CL for 10000 steps...


100%|███████████████████████████████████| 10000/10000 [00:05<00:00, 1720.11it/s]


Starting MCMC for g2 , M9 for 10000 steps...


...

100%|███████████████████████████████████| 10000/10000 [00:05<00:00, 1725.92it/s]


Starting MCMC for g14 , M9 for 10000 steps...


100%|███████████████████████████████████| 10000/10000 [00:05<00:00, 1728.46it/s]


Starting MCMC for g14 , CL for 10000 steps...


100%|███████████████████████████████████| 10000/10000 [00:05<00:00, 1704.57it/s]
